In [ ]:
import ast
import sys
from typing import Dict, List, Set

In [ ]:
class FunctionAnalyzer(ast.NodeVisitor):
    def __init__(self):
        self.functions = {}  # {function_name: {line_number, calls}}
        self.current_function = None
        self.function_order = []  # To maintain order of appearance
        self.inside_class = 0  # Track nesting level inside classes
        self.function_depth = 0  # Track nesting level inside functions
    
    def visit_ClassDef(self, node):
        """Visit class definitions and skip functions inside them"""
        self.inside_class += 1
        self.generic_visit(node)
        self.inside_class -= 1
    
    def visit_FunctionDef(self, node):
        """Visit function definitions (only top-level module functions)"""
        # Skip functions that are inside classes (methods) or nested inside other functions
        if self.inside_class > 0 or self.function_depth > 0:
            # Still need to increment depth to properly track nested functions
            self.function_depth += 1
            self.generic_visit(node)
            self.function_depth -= 1
            return
        
        func_name = node.name
        self.functions[func_name] = {
            'line_number': node.lineno,
            'calls': set()
        }
        self.function_order.append(func_name)
        
        # Set current function context for analyzing calls within it
        previous_function = self.current_function
        self.current_function = func_name
        
        # Increment function depth to track nested functions
        self.function_depth += 1
        
        # Visit the function body to find function calls
        self.generic_visit(node)
        
        # Restore previous state
        self.function_depth -= 1
        self.current_function = previous_function
    
    def visit_AsyncFunctionDef(self, node):
        """Visit async function definitions (same logic as regular functions)"""
        self.visit_FunctionDef(node)
    
    def visit_Call(self, node):
        """Visit function calls"""
        if self.current_function is not None:
            called_function = None
            
            # Handle direct function calls (e.g., func_name())
            if isinstance(node.func, ast.Name):
                called_function = node.func.id
                # Track the call if it's a function defined in this file
                if (called_function and 
                    called_function in self.functions and 
                    called_function != self.current_function):
                    self.functions[self.current_function]['calls'].add(called_function)
            
            # Handle attribute calls (e.g., self.method_name() or module.function())
            elif isinstance(node.func, ast.Attribute):
                # For now, we'll ignore attribute calls as they're typically methods
                # or external module functions
                pass
            
            # Check function arguments for function references (like in executor.submit(func, ...))
            for arg in node.args:
                if isinstance(arg, ast.Name):
                    arg_name = arg.id
                    # If this argument is a function defined in our file, it's being referenced/used
                    if (arg_name in self.functions and 
                        arg_name != self.current_function):
                        self.functions[self.current_function]['calls'].add(arg_name)
            
            # Also check keyword arguments
            for keyword in node.keywords:
                if isinstance(keyword.value, ast.Name):
                    kwarg_name = keyword.value.id
                    if (kwarg_name in self.functions and 
                        kwarg_name != self.current_function):
                        self.functions[self.current_function]['calls'].add(kwarg_name)
        
        self.generic_visit(node)

def analyze_python_file(file_path: str) -> Dict:
    """
    Analyze a Python file and extract function definitions and their dependencies
    
    Args:
        file_path (str): Path to the Python file to analyze
    
    Returns:
        Dict: Analysis results containing functions and their dependencies
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        
        # Parse the Python code into an AST
        tree = ast.parse(content)
        
        # Analyze the AST
        analyzer = FunctionAnalyzer()
        analyzer.visit(tree)
        
        return {
            'functions': analyzer.functions,
            'order': analyzer.function_order
        }
    
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
        return None
    except SyntaxError as e:
        print(f"Error: Syntax error in file '{file_path}': {e}")
        return None
    except Exception as e:
        print(f"Error: An unexpected error occurred: {e}")
        return None

def print_function_analysis(analysis_result: Dict, show_debug: bool = False):
    """
    Print the function analysis in the requested format
    
    Args:
        analysis_result (Dict): Result from analyze_python_file
        show_debug (bool): Whether to show debug information
    """
    if not analysis_result:
        return
    
    functions = analysis_result['functions']
    function_order = analysis_result['order']
    
    if show_debug:
        print("Debug Information:")
        print("-" * 40)
        for func_name in function_order:
            func_info = functions[func_name]
            print(f"Function '{func_name}' at line {func_info['line_number']}")
            print(f"  Raw calls: {func_info['calls']}")
        print()
    
    print("Function Analysis:")
    print("-" * 40)
    
    for func_name in function_order:
        func_info = functions[func_name]
        calls = func_info['calls']
        
        if calls:
            # # Sort the called functions for consistent output
            # calls = sorted(calls)
            calls_str = ", ".join(calls)
            print(f"{func_name} << {calls_str}")
        else:
            print(f"{func_name}")

In [ ]:
file_path = '../utils/nostra_add_product.py'
    
print(f"Analyzing file: {file_path}")
print("=" * 50)

# Analyze the file
result = analyze_python_file(file_path)

if result:
    print_function_analysis(result)

    # Additional statistics
    print("\n" + "-" * 40)
    print(f"Total functions found: {len(result['order'])}")

    # Show functions with no dependencies
    no_deps = [name for name in result['order'] 
              if not result['functions'][name]['calls']]
    if no_deps:
        print(f"Functions with no dependencies: {', '.join(no_deps)}")